# Анализ `processors-am4.json`

Таблица: **имя**, **цена**, **ядер**, **потоков**, **частота** (ГГц), **value** = потоки × частота, **value/$** = value / цена (цена в валюте из JSON, обычно BYN).

Сортировка по убыванию **value/$**.

Зависимости: `pip install pandas` (в среде Jupyter).

Файл `processors-am4.json` ищется в текущей рабочей папке или в `onliner.by-crawler/` (удобно, если kernel стартовал из корня репозитория).

In [17]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CANDIDATES = [
    ROOT / "processors-am4.json",
    ROOT / "onliner.by-crawler" / "processors-am4.json",
]
DATA = next((p for p in CANDIDATES if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Не найден processors-am4.json. Положите его рядом с ноутбуком или запустите краулер (run-search.bat)."
    )

with open(DATA, encoding="utf-8") as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
needed = ["name", "price", "coreCount", "threadCount", "maxFrequencyGHz"]
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise ValueError(f"В JSON нет колонок: {missing_cols}")

out = df[needed].copy()
out.columns = ["имя", "цена", "ядер", "потоков", "частота"]
out["value=потоков*частота"] = out["потоков"] * out["частота"]

price_num = pd.to_numeric(
    out["цена"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)
out["value/$"] = out["value=потоков*частота"].div(price_num).where(price_num > 150)

out_sorted = out.sort_values("value/$", ascending=False, na_position="last")
out_sorted

,имя,цена,ядер,потоков,частота,value=потоков*частота,value/$
29,AMD Ryzen 5 4500,245.13,6.0,12.0,4.1,49.2,0.200710
31,AMD Ryzen 5 5500,257.68,6.0,12.0,4.2,50.4,0.195591
27,AMD Ryzen 5 3600,271.69,6.0,12.0,4.2,50.4,0.185506
54,AMD Ryzen 7 5700,496.32,8.0,16.0,4.6,73.6,0.148291
59,AMD Ryzen 7 5700X,501.94,8.0,16.0,4.6,73.6,0.146631
...,...,...,...,...,...,...,...
8,AMD Athlon Pro 300GE,123.91,2.0,4.0,3.4,13.6,NaN
9,AMD Athlon X4 950,43.22,4.0,4.0,3.8,15.2,NaN
10,AMD Athlon X4 970,17.27,4.0,4.0,4.0,16.0,NaN
11,AMD Pro A6-9500E,60.89,NaN,NaN,3.4,NaN,NaN


In [18]:
# Строки без потоков или частоты (value не посчитать)
mask = out["потоков"].isna() | out["частота"].isna()
if mask.any():
    display(out.loc[mask])
else:
    print("Все строки имеют потоки и частоту для расчёта value.")

,имя,цена,ядер,потоков,частота,value=потоков*частота,value/$
1,AMD A10-9700,417.49,NaN,NaN,3.8,NaN,NaN
3,AMD A6-9500,17.90,NaN,NaN,3.8,NaN,NaN
4,AMD A6-9500E,228.41,NaN,NaN,3.4,NaN,NaN
11,AMD Pro A6-9500E,60.89,NaN,NaN,3.4,NaN,NaN


In [19]:
# Поиск по подстроке в имени (удобно, если таблица свёрнута в `..` в редакторе)
SUBSTR = "X4 950"
out_sorted[out_sorted["имя"].str.contains(SUBSTR, case=False, na=False)]

,имя,цена,ядер,потоков,частота,value=потоков*частота,value/$
9,AMD Athlon X4 950,43.22,4.0,4.0,3.8,15.2,NaN
